# CIFAR-100: ConvNeXtV2-Pico vs EfficientNet-B0

Локальный тест двух моделей перед переходом на FL.  
Данные: `data/cifar100` (HuggingFace datasets, 50k train / 10k test, 100 классов, 32×32).  
Модели принимают 224×224 (pretrained ImageNet) → resize в трансформах.

In [ ]:
# Установка timm (нужен для ConvNeXtV2)
!pip install timm -q

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_from_disk
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from fl_app.models.cifar100 import build_model, MODEL_CONFIGS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"torch  : {torch.__version__}")
print()
print("Доступные модели:")
for name, cfg in MODEL_CONFIGS.items():
    print(f"  {name:<20s} — {cfg['description']}")

## 1. Конфигурация

In [ ]:
# ── выбор модели ─────────────────────────────────────────────────
MODEL_NAME = "wrn_28_4"   # "wrn_28_4" | "efficientnet_b0" | "cct_7_3x1"

cfg = MODEL_CONFIGS[MODEL_NAME]   # дефолтные гиперпараметры из фабрики

# ── данные ───────────────────────────────────────────────────────
DATA_DIR    = "data/cifar100"
NUM_CLASSES = 100
IMAGE_SIZE  = 32

# ── обучение (можно переопределить cfg) ──────────────────────────
NUM_EPOCHS   = cfg["epochs"]
BATCH_SIZE   = cfg["batch_size"]
LR           = cfg["lr"]
WEIGHT_DECAY = cfg["weight_decay"]
LABEL_SMOOTH = cfg["label_smooth"]
MIXUP_ALPHA  = cfg["mixup_alpha"]
NUM_WORKERS  = 4
USE_AMP      = True

# ── сохранение ───────────────────────────────────────────────────
SAVE_DIR = f"runs/cifar100_local/{MODEL_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Model    : {MODEL_NAME}")
print(f"Config   : {cfg['description']}")
print(f"Epochs   : {NUM_EPOCHS}  |  BS: {BATCH_SIZE}  |  LR: {LR}  |  WD: {WEIGHT_DECAY}")
print(f"Mixup    : {MIXUP_ALPHA}  |  AMP: {USE_AMP}")
print(f"Save dir : {SAVE_DIR}")

## 2. Датасет и даталоадер

In [ ]:
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

# слабая аугментация 
train_tf = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMAGE_SIZE, padding=IMAGE_SIZE // 8),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# сильная аугментация
# train_tf = transforms.Compose([
#     transforms.Resize(IMAGE_SIZE),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(IMAGE_SIZE, padding=IMAGE_SIZE // 8),
#     transforms.RandAugment(num_ops=3, magnitude=12),
#     transforms.ToTensor(),
#     transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
#     transforms.RandomErasing(p=0.25),
# ])

val_tf = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])


class CIFAR100HF(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img    = sample["img"].convert("RGB")
        label  = sample["fine_label"]
        return self.transform(img), label


hf_ds = load_from_disk(DATA_DIR)
CLASS_NAMES = hf_ds["train"].features["fine_label"].names

train_ds = CIFAR100HF(hf_ds["train"], train_tf)
val_ds   = CIFAR100HF(hf_ds["test"],  val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Val  : {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"Classes: {NUM_CLASSES}  (e.g. {CLASS_NAMES[:5]} ...)")

## 3. Инициализация модели

In [ ]:
model = build_model(MODEL_NAME, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model        : {MODEL_NAME}")
print(f"Total params : {total_params:,}  ({total_params/1e6:.1f}M)")

dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape : {out.shape}  ✓")

## 4. Параметры обучения

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

if cfg["optimizer"] == "sgd":
    optimizer = torch.optim.SGD(
        model.parameters(), lr=LR,
        momentum=cfg.get("momentum", 0.9),
        weight_decay=WEIGHT_DECAY, nesterov=True,
    )
else:  # adamw
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
    )

if cfg["scheduler"] == "multistep":
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=cfg["milestones"], gamma=cfg["gamma"],
    )
    sched_desc = f"MultiStepLR(milestones={cfg['milestones']}, gamma={cfg['gamma']})"
else:  # cosine
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6,
    )
    sched_desc = f"CosineAnnealingLR(T_max={NUM_EPOCHS})"

print(f"Optimizer : {cfg['optimizer'].upper()}(lr={LR}, wd={WEIGHT_DECAY})")
print(f"Scheduler : {sched_desc}")

4.5 Optuna: автоподбор гиперпараметров

In [ ]:
import optuna

OPTUNA_EPOCHS = 100
OPTUNA_TRIALS = 10

def objective(trial):
    start_time = time.time()

    # ---------------------------------
    # Подбираем только lr и weight_decay
    # ---------------------------------
    lr = trial.suggest_float("lr", 0.07, 0.13)
    wd = trial.suggest_float("weight_decay", 2e-4, 1e-3, log=True)

    # baseline фиксируем
    momentum = 0.9
    milestones = [30, 60, 80]
    gamma = 0.2
    mixup_a = 0.4
    label_sm = 0.1
    drop_rate = 0.3

    model = build_model(
        MODEL_NAME,
        num_classes=NUM_CLASSES,
        drop_rate=drop_rate
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_sm)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=momentum,
        weight_decay=wd,
        nesterov=True
    )

    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=milestones,
        gamma=gamma
    )

    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_val_acc = 0.0

    for epoch in range(OPTUNA_EPOCHS):
        model.train()
        train_loss_sum = 0.0
        train_samples = 0

        for imgs, labels in train_loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                if mixup_a > 1e-8:
                    lam = np.random.beta(mixup_a, mixup_a)
                    idx = torch.randperm(imgs.size(0), device=imgs.device)
                    mixed_imgs = lam * imgs + (1.0 - lam) * imgs[idx]

                    logits = model(mixed_imgs)
                    loss = (
                        lam * criterion(logits, labels) +
                        (1.0 - lam) * criterion(logits, labels[idx])
                    )
                else:
                    logits = model(imgs)
                    loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = labels.size(0)
            train_loss_sum += loss.item() * bs
            train_samples += bs

        scheduler.step()

        model.eval()
        correct = 0
        total = 0
        val_loss_sum = 0.0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    logits = model(imgs)
                    loss = criterion(logits, labels)

                preds = logits.argmax(dim=1)

                bs = labels.size(0)
                val_loss_sum += loss.item() * bs
                correct += (preds == labels).sum().item()
                total += bs

        val_acc = correct / total if total > 0 else 0.0
        best_val_acc = max(best_val_acc, val_acc)

        avg_train_loss = train_loss_sum / train_samples if train_samples > 0 else math.nan
        avg_val_loss = val_loss_sum / total if total > 0 else math.nan
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"[Trial {trial.number:02d}] "
            f"Epoch {epoch + 1:03d}/{OPTUNA_EPOCHS} | "
            f"lr={current_lr:.5f} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"val_loss={avg_val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"best={best_val_acc:.4f}"
        )

        trial.report(best_val_acc, step=epoch)

        if trial.should_prune():
            print(f"[Trial {trial.number:02d}] pruned at epoch {epoch + 1}")
            raise optuna.TrialPruned()

    elapsed = time.time() - start_time
    print(f"[Trial {trial.number:02d}] finished in {elapsed/60:.2f} min")

    return best_val_acc


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=3,
        n_warmup_steps=15,
        interval_steps=1
    ),
)

study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print(f"\n{'='*60}")
print(f"Лучшая accuracy: {study.best_value * 100:.2f}%")
print("Лучшие параметры:")
for k, v in study.best_params.items():
    print(f"  {k:<20s} = {v}")
print(f"{'='*60}")

## 5. Обучение

In [ ]:
import sys
from copy import deepcopy

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_val_acc = 0.0
best_ckpt    = os.path.join(SAVE_DIR, "best.pt")
log_path     = os.path.join(SAVE_DIR, "train.log")

N_TRAIN = len(train_loader)
N_VAL   = len(val_loader)
scaler  = torch.amp.GradScaler("cuda", enabled=USE_AMP)

class EMA:
    def __init__(self, model, decay=0.999):
        self.ema_model = deepcopy(model)
        self.decay = decay
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.ema_model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)

ema = EMA(model, decay=0.999)

header = f"{'Epoch':>5}  {'T-Loss':>8}  {'T-Acc':>7}  {'V-Loss':>8}  {'V-Acc':>7}  {'V-F1':>7}  {'LR':>8}  {'Time':>6}"
sep    = "-" * len(header)

def log(line: str, file=None):
    print(line)
    if file:
        file.write(line + "\n")
        file.flush()


def mixup_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

def cutmix_data(x, y, alpha):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    B, C, H, W = x.shape
    cut_ratio = np.sqrt(1.0 - lam)
    rw, rh = int(W * cut_ratio), int(H * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - rw // 2, 0, W)
    x2 = np.clip(cx + rw // 2, 0, W)
    y1 = np.clip(cy - rh // 2, 0, H)
    y2 = np.clip(cy + rh // 2, 0, H)
    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x_cut, y, y[idx], lam


def train_epoch():
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for i, (imgs, labels) in enumerate(train_loader, 1):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            if MIXUP_ALPHA > 0:
                if np.random.rand() < 0.5:
                    imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                else:
                    imgs, y_a, y_b, lam = cutmix_data(imgs, labels, MIXUP_ALPHA)
                logits = model(imgs)
                loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
                preds  = logits.argmax(1)
                correct += (lam*(preds==y_a).float() + (1-lam)*(preds==y_b).float()).sum().item()
            else:
                logits = model(imgs)
                loss   = criterion(logits, labels)
                correct += (logits.argmax(1) == labels).sum().item()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)
        total_loss += loss.item() * len(labels)
        total      += len(labels)
        print(f"  train {i:>4}/{N_TRAIN}  loss={total_loss/total:.4f}  acc={correct/total*100:.1f}%",
              end="\r", flush=True)
    print(" " * 60, end="\r")
    return total_loss / total, correct / total


def val_epoch():
    ema.ema_model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(val_loader, 1):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            preds = logits.argmax(1)
            total_loss += loss.item() * len(labels)
            correct    += (preds == labels).sum().item()
            total      += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            print(f"  val   {i:>4}/{N_VAL}  loss={total_loss/total:.4f}  acc={correct/total*100:.1f}%",
                  end="\r", flush=True)
    print(" " * 60, end="\r")
    _, _, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    return total_loss / total, correct / total, f1


with open(log_path, "w") as f:
    log(f"Model: {MODEL_NAME}  |  {cfg['description']}", f)
    log(f"Epochs: {NUM_EPOCHS}  BS: {BATCH_SIZE}  LR: {LR}  WD: {WEIGHT_DECAY}", f)
    log("", f)
    log(header, f)
    log(sep, f)

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_acc         = train_epoch()
        vl_loss, vl_acc, vl_f1 = val_epoch()

        scheduler.step()
        elapsed    = time.time() - t0
        current_lr = scheduler.get_last_lr()[0]

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)
        history["val_f1"].append(vl_f1)

        marker = " ★" if vl_acc > best_val_acc else ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_ckpt)

        line = (f"{epoch:>5}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  "
                f"{vl_loss:>8.4f}  {vl_acc*100:>6.2f}%  {vl_f1:.4f}  "
                f"{current_lr:>8.1e}  {elapsed:>5.0f}s{marker}")
        log(line, f)

    log("", f)
    log(f"Best val acc: {best_val_acc*100:.2f}%  → {best_ckpt}", f)

print()
print(f"Best val acc: {best_val_acc*100:.2f}%")
print(f"Log saved  → {log_path}")

## 6. Оценка качества (best checkpoint)

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="macro", zero_division=0
)
prec_per, rec_per, f1_per, support = precision_recall_fscore_support(
    all_labels, all_preds, average=None, zero_division=0
)

print(f"Model       : {MODEL_NAME}")
print(f"Accuracy    : {acc*100:.2f}%")
print(f"Precision   : {prec_macro*100:.2f}%  (macro)")
print(f"Recall      : {rec_macro*100:.2f}%  (macro)")
print(f"F1          : {f1_macro:.4f}  (macro)")
print()

worst_idx = np.argsort(f1_per)[:5]
print("Top-5 худших классов по F1:")
for i in worst_idx:
    print(f"  [{i:>3}] {CLASS_NAMES[i]:<20s}  F1={f1_per[i]:.3f}  "
          f"P={prec_per[i]:.3f}  R={rec_per[i]:.3f}  n={support[i]}")

print()
best_idx = np.argsort(f1_per)[-5:][::-1]
print("Top-5 лучших классов по F1:")
for i in best_idx:
    print(f"  [{i:>3}] {CLASS_NAMES[i]:<20s}  F1={f1_per[i]:.3f}  "
          f"P={prec_per[i]:.3f}  R={rec_per[i]:.3f}  n={support[i]}")

## 7. Графики (Loss / Accuracy / F1)

In [ ]:
model_label = {MODEL_NAME}
epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f"{model_label} — CIFAR-100 (best val acc: {best_val_acc*100:.2f}%)", fontsize=13)

ax = axes[0]
ax.plot(epochs, history["train_loss"], label="train", marker=".")
ax.plot(epochs, history["val_loss"],   label="val",   marker=".")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(epochs, [v*100 for v in history["train_acc"]], label="train", marker=".")
ax.plot(epochs, [v*100 for v in history["val_acc"]],   label="val",   marker=".")
ax.set_title("Accuracy")
ax.set_xlabel("Epoch"); ax.set_ylabel("Acc (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(epochs, history["val_f1"], label="val macro-F1", marker=".", color="green")
ax.set_title("Macro F1 (val)")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "curves.png")
plt.savefig(plot_path, dpi=120)
plt.show()
print(f"Saved → {plot_path}")

## 8. Per-class accuracy

In [ ]:
model_label = {MODEL_NAME}

per_class_acc = np.array([
    (all_preds[all_labels == c] == c).mean() if (all_labels == c).sum() > 0 else 0.0
    for c in range(NUM_CLASSES)
])

sorted_idx = np.argsort(per_class_acc)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle(f"{model_label} — Per-class Accuracy (CIFAR-100)", fontsize=13)

ax = axes[0]
colors = ["#d62728" if per_class_acc[i] < 0.4 else
          "#ff7f0e" if per_class_acc[i] < 0.6 else
          "#2ca02c" for i in sorted_idx]
ax.bar(range(NUM_CLASSES), per_class_acc[sorted_idx] * 100, color=colors, width=0.8)
ax.set_title("Все классы (sorted)")
ax.set_xlabel("Класс (ранг)"); ax.set_ylabel("Accuracy (%)")
ax.axhline(per_class_acc.mean() * 100, color="navy", linestyle="--", linewidth=1.2,
           label=f"mean {per_class_acc.mean()*100:.1f}%")
ax.legend(); ax.grid(True, axis="y", alpha=0.3); ax.set_xticks([])

ax = axes[1]
n = 20
worst_n = sorted_idx[:n]
ax.barh(np.arange(n), per_class_acc[worst_n] * 100, color="#d62728", alpha=0.8)
ax.set_yticks(np.arange(n))
ax.set_yticklabels([CLASS_NAMES[i] for i in worst_n], fontsize=9)
ax.set_title(f"Топ-{n} худших классов")
ax.set_xlabel("Accuracy (%)")
ax.axvline(per_class_acc.mean() * 100, color="navy", linestyle="--", linewidth=1.2)
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "per_class_acc.png")
plt.savefig(plot_path, dpi=120)
plt.show()
print(f"Saved → {plot_path}")
print(f"Mean per-class acc: {per_class_acc.mean()*100:.2f}%")
print(f"Std  per-class acc: {per_class_acc.std()*100:.2f}%")